In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
from collections import Counter
from pathlib import Path
import os

In [5]:
# -----------------------------
# CONFIG
# -----------------------------
REPO_ROOT = Path(os.getcwd()).parent 

VIDEO_PATH = "/work/classtmp/FIRST-Robotics-Competition-Data-Challenge-Videos/cropped_videos/cropped_Qualification 45 - 2025 Central Missouri Regional.mp4"

ROBOT_MODEL_PATH = REPO_ROOT / "yolov8_model" / "best_tuned_yolov8.pt"
NUMBER_MODEL_PATH = REPO_ROOT / "number_reading" / "best.pt"

ROBOT_CLASS_ID = 1
RED_NUMBER_CLASS_ID = 1
BLUE_NUMBER_CLASS_ID = 0

FRAME_SKIP = 15         # process every Nth frame

In [6]:
# -----------------------------
# LOAD MODELS
# -----------------------------
robot_model = YOLO(ROBOT_MODEL_PATH)
number_model = YOLO(NUMBER_MODEL_PATH)

reader = easyocr.Reader(['en'], gpu=True)

FileNotFoundError: [Errno 2] No such file or directory: '/work/classtmp/kmwendl/FIRST-Robotics-Competition-Data-Challenge/number_reading/best.pt'

In [4]:
# -----------------------------
# IMAGE PROCESSING FUNCTIONS
# -----------------------------
def estimate_angle_from_crop(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return 0.0

    cnt = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)
    angle = rect[-1]

    if angle < -45:
        angle += 90

    return angle


def rotate_image(img, angle):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(
        img,
        M,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )


def preprocess_crop(crop, threshold):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    angle = estimate_angle_from_crop(crop)
    rotated = rotate_image(thresh, angle)

    return rotated

def preprocess_variants(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    variants = []

    # 1. Simple thresholds
    for t in [150, 165, 180, 195, 210]:
        _, th = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
        variants.append(th)

    # 2. Adaptive threshold
    adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    variants.append(adaptive)

    # 3. Otsu
    _, otsu = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    variants.append(otsu)

    # 4. Histogram equalization + Otsu
    eq = cv2.equalizeHist(gray)
    _, th_eq = cv2.threshold(eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    variants.append(th_eq)

    return variants

In [5]:
# -----------------------------
# OCR
# -----------------------------
def read_number_from_image(img):
    variants = preprocess_variants(img)
    guesses = []

    angle = estimate_angle_from_crop(img)
    for processed in variants:
        rotated = rotate_image(processed, angle)

        results = reader.readtext(
            rotated,
            allowlist="0123456789",
            detail=1,
            paragraph=False
        )

        # Filter low confidence
        results = [r for r in results if r[2] > 0.5]

        if results:
            guesses.append(results[0][1])

    if not guesses:
        return None

    c = Counter(guesses)
    max_count = max(c.values())

    tied = [num for num, count in c.items() if count == max_count]

    return max(tied, key=lambda x: len(str(x)))

In [6]:
# -----------------------------
# MATCHING SYSTEM
# -----------------------------
def levenshtein(a, b):
    a, b = str(a), str(b)
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]

    for i in range(len(a)+1):
        dp[i][0] = i
    for j in range(len(b)+1):
        dp[0][j] = j

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[-1][-1]


def similarity(a, b):
    if not a or not b:
        return 0
    dist = levenshtein(a, b)
    return 1 - dist / max(len(str(a)), len(str(b)))


def alignment_score(a, b):
    a, b = str(a), str(b)
    best = 0

    for shift in range(-len(b), len(a)+1):
        matches = 0
        for i in range(len(a)):
            j = i - shift
            if 0 <= j < len(b) and a[i] == b[j]:
                matches += 1
        best = max(best, matches)

    return best / max(len(a), len(b))


def combined_score(a, b, w1=0.7, w2=0.3):
    return w1 * similarity(a, b) + w2 * alignment_score(a, b)


def match_number_single(detected, nums):
    if detected is None:
        return None

    best_score = 0.5
    match = None

    for n in nums:
        current = combined_score(detected, n)
        if current > best_score:
            best_score = current
            match = n

    return match

In [7]:
# -----------------------------
# MAIN PIPELINE
# -----------------------------
def process_video(video_path, red_team_numbers=[], blue_team_numbers=[], crop=True):
    cap = cv2.VideoCapture(video_path)

    frame_idx = 0
    results = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        if frame_idx % FRAME_SKIP != 0:
            continue

        # Crop bottom
        h, w = frame.shape[:2]
        if crop:
            frame = frame[int(h * 3 / 5):h, 0:w]

        robot_results = robot_model(frame, verbose=False)[0]

        frame_data = []

        for box in robot_results.boxes:
            if int(box.cls[0]) != ROBOT_CLASS_ID:
                continue

            if box.conf[0] < 0.4:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            robot_crop = frame[y1:y2, x1:x2]

            if robot_crop.size == 0:
                continue

            number_results = number_model(robot_crop, verbose=False)[0]

            for nbox in number_results.boxes:
                cls_id = int(nbox.cls[0])
                if cls_id == RED_NUMBER_CLASS_ID:
                    team_list = red_team_numbers
                elif cls_id == BLUE_NUMBER_CLASS_ID:
                    team_list = blue_team_numbers
                else:
                    continue

                if nbox.conf[0] < 0.4:
                    continue

                nx1, ny1, nx2, ny2 = map(int, nbox.xyxy[0])

                # Padding
                pad = 5
                h2, w2 = robot_crop.shape[:2]
                nx1 = max(0, nx1 - pad)
                ny1 = max(0, ny1 - pad)
                nx2 = min(w2, nx2 + pad)
                ny2 = min(h2, ny2 + pad)

                # Filter tiny boxes
                if (nx2 - nx1) < 30 or (ny2 - ny1) < 15:
                    continue

                number_crop = robot_crop[ny1:ny2, nx1:nx2]

                if number_crop.size == 0:
                    continue

                detected = read_number_from_image(number_crop)
                matched = match_number_single(detected, team_list)

                frame_data.append({
                    "detected": detected,
                    "matched": matched
                })

        print(f"Frame {frame_idx}: {frame_data}")

        results.append({
            "frame": frame_idx,
            "detections": frame_data
        })

    cap.release()
    return results

In [8]:
# -----------------------------
# PRINT DETECTIONS EACH FRAME
# -----------------------------
RED_TEAM_NUMBERS = [8825, 1736, 2357]
BLUE_TEAM_NUMBERS = [5809, 9570, 3928]
if __name__ == "__main__":
    output = process_video(VIDEO_PATH, RED_TEAM_NUMBERS, BLUE_TEAM_NUMBERS, False)

    print("\nFinal Results:")
    print(output)

Frame 15: []
Frame 30: []
Frame 45: []
Frame 60: []
Frame 75: []
Frame 90: []
Frame 105: []
Frame 120: [{'detected': None, 'matched': None}]
Frame 135: []
Frame 150: []
Frame 165: []
Frame 180: []
Frame 195: []
Frame 210: []
Frame 225: []
Frame 240: []
Frame 255: []
Frame 270: [{'detected': None, 'matched': None}]
Frame 285: [{'detected': None, 'matched': None}, {'detected': None, 'matched': None}]
Frame 300: [{'detected': '4734', 'matched': None}, {'detected': '3928', 'matched': 3928}, {'detected': '3928', 'matched': 3928}, {'detected': None, 'matched': None}]
Frame 315: [{'detected': '3028', 'matched': 3928}, {'detected': '1736', 'matched': 1736}]
Frame 330: [{'detected': None, 'matched': None}, {'detected': None, 'matched': None}, {'detected': '877', 'matched': None}, {'detected': '2', 'matched': None}]
Frame 345: [{'detected': None, 'matched': None}, {'detected': None, 'matched': None}]
Frame 360: [{'detected': '17', 'matched': None}, {'detected': '1176', 'matched': None}]
Frame 37

In [10]:
def count_matches(results):
    none_count = 0
    match_count = 0

    for frame in results:
        for det in frame["detections"]:
            if det["matched"] is None:
                none_count += 1
            else:
                match_count += 1

    return none_count, match_count


none_count, match_count = count_matches(output)

print(f"Matched: {match_count}")
print(f"None: {none_count}")

Matched: 293
None: 817
